In [1]:
import pandas as pd
from sodapy import Socrata
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)

In [2]:
import requests
import gspread
from oauth2client.service_account import ServiceAccountCredentials

In [3]:
def getXrefs():
    '''Reads the Inventory google sheet and gets the datasets title and cross-refs it to the Socrata 4x4 id.  Also
    gets the fields by 4x4 dataset id and by the title'''
    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
             "https://www.googleapis.com/auth/drive.file",
                  "https://www.googleapis.com/auth/drive"]

    creds = ServiceAccountCredentials.from_json_keyfile_name('/home/joe/work/client_secret.json',
     scope)
    client = gspread.authorize(creds)

    gc = gspread.service_account("/home/joe/work/client_secret.json")
    # for gg in gc.list_spreadsheet_files():
    #      print("GGGGG ",gg)
    
    tracker = client.open('BIC Dataset Tracker').worksheet(
    'PublishedData')
   

    df = pd.DataFrame(tracker.get_all_records(head=3))

    return df
            
df = getXrefs()

GSpreadException: the header row in the worksheet is not unique, try passing 'expected_headers' to get_all_records

In [6]:
df.columns

# for col in sorted(df.columns):
#     print(col)

Index(['Dataset Title', 'Short Description', 'Category', 'Keywords', 'Type',
       'License Type', 'Data Provider', 'Data Provided by', 'Source Link',
       'State Steward', 'Citation', 'Agency Program Page',
       'Agency Data Series Page', 'Business Contact and Phone',
       'Technical Contact and Phone', 'Data Source', 'Unit of Analysis',
       'Granularity Coverage', 'Geographic Extent and Division',
       'Collection Mode', 'Collection Methodology',
       'Data Collection Instrument', 'Date of Initial Dataset Creation',
       'Field Names, comma delimited', 'Oldest Record in Dataset',
       'Newest Record in Dataset', 'Long Description', 'Data Dictionary',
       'Additional Metadata', 'Technical Documentation',
       'Data Quality Certification',
       'Applicable Information Quality Guideline Designation',
       'Stewardship Plan', 'Collection Method', 'Horizontal Accuracy',
       'Horizontal Coordinate System', 'Update Schedule', 'Update Method',
       'Source Upd

In [8]:
len(df["Socrata Link"].to_list())

399

In [ ]:
## I set this up to check the logic of the metadata_updater using the new dataset tracker
for w4x4 in df["Socrata Link"]:
    single_row = df.loc[df["Socrata Link"] == w4x4]
    tag_list = str(single_row["Keywords"].item())
    tags = tag_list.split(",")
    print(w4x4)
    data = {"name": single_row["Dataset Title"].item(),
                "category": single_row["Category"].item(),
                "attribution": single_row["State Steward"].item(),
                "license": str(single_row["License Type"].item()),
                "attributionLink": single_row["Agency Program Page"].item(),
                "description": single_row["Short Description"].item(),
                "tags": tags,
                "customFields": {
                    "Data Updates": {
                        "Update Schedule": single_row["Update Schedule"].item(),
                        "Update Method": single_row["Update Method"].item(),
                        "Update Type": single_row["Update Type"].item(),
                        "Source Update Schedule": single_row["Source Update Schedule"].item(),
                        "Total Records At Initial Publish": single_row["Total Records at Initial Publish"].item()
                    },
                    "Dataset Coverage": {
                        "Unit of Analysis": single_row["Unit of Analysis"].item(),
                        "Granularity": single_row["Granularity Coverage"].item(),
                        "Geographic Coverage": single_row["Geographic Extent and Division"].item()
                        },
                    "Geospatial": {
                        "Collection Method": single_row["Collection Method"].item(),
                        "Horizontal Accuracy": single_row["Horizontal Accuracy"].item(),
                        "Horizontal Coordinate System": single_row["Horizontal Coordinate System"].item(),
                        "Web Display Coordinate System": single_row["Web Display Coordinate System"].item(),
                        "Coordinate System Disclaimer": single_row["Coordinate System Disclaimer"].item()
                    },  
                    "Additional Dataset Documentation": {
                        "Data Dictionary": single_row["Data Dictionary"].item(),
                        "Additional Metadata": single_row["Additional Metadata"].item(),
                        "Technical Documentation": single_row["Technical Documentation"].item()
                    },
                    "Data Description": {
                        "Single Row": single_row["Single Row"].item(),
                        "Long Description": single_row["Long Description"].item(),
                        "Collection Mode": single_row["Collection Mode"].item(),
                    #    "Collection Method": single_row["Collection Methodology"].item(),
                        "Data Collection Instrument": single_row["Data Collection Instrument"].item(),
                        "Date of Initial Dataset Creation": single_row["Date of Initial Dataset Creation"].item(),
                        "Field Names, comma delimited": single_row["Field Names, comma delimited"].item(),
                        "Oldest Record in Dataset": single_row["Oldest Record in Dataset"].item(),
                        "Newest Record in Dataset": single_row["Newest Record in Dataset"].item()
                    },
                    "Data Quality": {
                        "Expected Update Frequency": single_row["Update Type"].item()
                    },
                    "Contributing Agency Information": {
                        "Citation": single_row["Citation"].item(),
                        "Agency Program Page": single_row["Agency Program Page"].item(),
                        "Agency Data Series Page": single_row["Agency Data Series Page"].item(),
                        "Data Source": single_row["Data Source"].item()
                    },
                }}
    putHist(data)

In [28]:
hist={}
def putHist(data):
    for k,v in data.items():
       # print(k,type(v))
        if isinstance(v,list):
            # for vals in v:
            #     print("   ",type(vals),vals)
            if k not in hist:
                hist[k] = []
            hist[k].append(v)
        elif isinstance(v,dict):
            for k2,v2 in v.items():
                # print("   ",k2,type(v2),v2)
                if isinstance(v2,dict):
                   for k3,v3 in v2.items():
                       # print("      ",k3,type(v3),v3)
                       if k3 not in hist:
                          hist[k3] = []
                       hist[k3].append(v3)
                else:
                    if k2 not in hist:
                        hist[k2] = []
                    hist[k2].append(v2)
        else:
            if k not in hist:
                hist[k] = []
            hist[k].append(v)

In [31]:
for key,val in hist.items():
    print(key,len(val))

name 399
category 399
attribution 399
license 399
attributionLink 399
description 399
tags 399
Update Schedule 399
Update Method 399
Update Type 399
Source Update Schedule 399
Total Records At Initial Publish 399
Unit of Analysis 399
Granularity 399
Geographic Coverage 399
Collection Method 399
Horizontal Accuracy 399
Horizontal Coordinate System 399
Web Display Coordinate System 399
Coordinate System Disclaimer 399
Data Dictionary 399
Additional Metadata 399
Technical Documentation 399
Single Row 399
Long Description 399
Collection Mode 399
Data Collection Instrument 399
Date of Initial Dataset Creation 399
Field Names, comma delimited 399
Oldest Record in Dataset 399
Newest Record in Dataset 399
Expected Update Frequency 399
Citation 399
Agency Program Page 399
Agency Data Series Page 399
Data Source 399


In [33]:
tmp = pd.DataFrame(hist)

In [35]:
for col in tmp.columns:
    if col != "tags":
        print(col,tmp[col].nunique())

name 399
category 24
attribution 37
license 3
attributionLink 75
description 294
Update Schedule 9
Update Method 6
Update Type 10
Source Update Schedule 8
Total Records At Initial Publish 183
Unit of Analysis 11
Granularity 3
Geographic Coverage 14
Collection Method 7
Horizontal Accuracy 5
Horizontal Coordinate System 23
Web Display Coordinate System 4
Coordinate System Disclaimer 2
Data Dictionary 10
Additional Metadata 31
Technical Documentation 4
Single Row 236
Long Description 245
Collection Mode 38
Data Collection Instrument 29
Date of Initial Dataset Creation 58
Field Names, comma delimited 220
Oldest Record in Dataset 92
Newest Record in Dataset 68
Expected Update Frequency 10
Citation 37
Agency Program Page 75
Agency Data Series Page 110
Data Source 25


In [11]:
expectedColumns=['DatasetTitle', 'Short Description', 'Category', 'Keywords', 'Type',
       'License Type', 'Data Provider', 'Data Provided by', 'Source Link',
       'State Steward', 'Citation', 'Agency Program Page',
       'Agency Data Series Page', 'Business Contact and Phone',
       'Technical Contact and Phone', 'Data Source', 'Unit of Analysis',
       'Granularity Coverage', 'Geographic Extent and Division',
       'Collection Mode', 'Collection Methodology',
       'Data Collection Instrument', 'Date of Initial Dataset Creation',
       'Field Names, comma delimited', 'Oldest Record in Dataset',
       'Newest Record in Dataset', 'Long Description', 'Data Dictionary',
       'Additional Metadata', 'Technical Documentation',
       'Data Quality Certification',
       'Applicable Information Quality Guideline Designation',
       'Stewardship Plan', 'Collection Method', 'Horizontal Accuracy',
       'Horizontal Coordinate System', 'Update Schedule', 'Update Method',
       'Source Update Schedule', 'Update Type',
       'Total Records at Initial Publish', 'Row Class RDF',
       'Subject Column RDF', 'Single Row', 'Row Count (11/20/17)',
       'Total Fields at Initial Publish',
       'FIle Size at Initial Publish or as of 3-1-2016',
       'Expected approximate increase in record count at update',
       'Date Published to CIM', 'GoCode FY Published to CIM', 'Socrata Link',
       'API 4x4', 'Web Display Coordinate System',
       'Coordinate System Disclaimer', 'Related Datasets',
       'Quarter of Gov FY Published', 'CIM Updated', 'Days Since CIM Update',
       'cimAllData Updates', 'Complexity']

In [12]:
expectedColumns

['DatasetTitle',
 'Short Description',
 'Category',
 'Keywords',
 'Type',
 'License Type',
 'Data Provider',
 'Data Provided by',
 'Source Link',
 'State Steward',
 'Citation',
 'Agency Program Page',
 'Agency Data Series Page',
 'Business Contact and Phone',
 'Technical Contact and Phone',
 'Data Source',
 'Unit of Analysis',
 'Granularity Coverage',
 'Geographic Extent and Division',
 'Collection Mode',
 'Collection Methodology',
 'Data Collection Instrument',
 'Date of Initial Dataset Creation',
 'Field Names, comma delimited',
 'Oldest Record in Dataset',
 'Newest Record in Dataset',
 'Long Description',
 'Data Dictionary',
 'Additional Metadata',
 'Technical Documentation',
 'Data Quality Certification',
 'Applicable Information Quality Guideline Designation',
 'Stewardship Plan',
 'Collection Method',
 'Horizontal Accuracy',
 'Horizontal Coordinate System',
 'Update Schedule',
 'Update Method',
 'Source Update Schedule',
 'Update Type',
 'Total Records at Initial Publish',
 'Row 

In [13]:
set(expectedColumns)-set(df.columns)

{'DatasetTitle'}

In [14]:
set(df.columns)-set(expectedColumns)

{'Dataset Title'}